# Embed Products → Cloud SQL (GPU)

Computes BGE-M3 embeddings for all products in Cloud SQL and writes them back.
Uses T4 GPU (~15 min) instead of MacBook CPU (~4+ hours).

**Prerequisites:**
- Cloud SQL `chatbeauty-db` instance with products table populated
- BGE-M3 model on Google Drive or GCS
- Colab runtime set to **T4 GPU**

## 0. Setup

In [ ]:
!pip install -q sentence-transformers cloud-sql-python-connector psycopg2-binary pg8000

In [ ]:
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()

In [ ]:
# === CONFIGURE THESE ===
PROJECT_ID = "your-project-id"                    # <-- CHANGE
REGION = "asia-northeast3"
INSTANCE_NAME = "chatbeauty-db"
DB_NAME = "chatbeauty"
DB_USER = "postgres"
DB_PASSWORD = "your-password"                      # <-- CHANGE (use raw @, not %40)

INSTANCE_CONNECTION_NAME = f"{PROJECT_ID}:{REGION}:{INSTANCE_NAME}"

# Model path on Google Drive
MODEL_PATH = "/content/drive/MyDrive/ChatBeauty/ml/model/retrieval/bge-m3-finetuned-20260202-120852"

BATCH_SIZE = 256

## 1. Connect to Cloud SQL

In [ ]:
from google.cloud.sql.connector import Connector

connector = Connector()

def get_conn():
    return connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pg8000",
        user=DB_USER,
        password=DB_PASSWORD,
        db=DB_NAME,
    )

# Test connection
conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM products")
total = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM products WHERE embedding IS NOT NULL")
with_emb = cur.fetchone()[0]
print(f"Products: {total} total, {with_emb} with embeddings, {total - with_emb} to embed")
cur.close()
conn.close()

## 2. Load Model & Fetch Products

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"Loading model from {MODEL_PATH}...")
model = SentenceTransformer(MODEL_PATH)
print(f"Model loaded. Device: {model.device}")

In [ ]:
# Fetch all products that need embeddings
conn = get_conn()
cur = conn.cursor()
cur.execute("""
    SELECT parent_asin, embedding_text
    FROM products
    WHERE embedding IS NULL AND embedding_text IS NOT NULL
    ORDER BY parent_asin
""")
rows = cur.fetchall()
cur.close()
conn.close()

asins = [r[0] for r in rows]
texts = [r[1] for r in rows]
print(f"Fetched {len(texts)} products to embed")

## 3. Encode Embeddings (GPU)

In [ ]:
import time

start = time.time()
all_embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)
elapsed = time.time() - start
print(f"Encoded {len(texts)} products in {elapsed:.1f}s ({len(texts)/elapsed:.0f} texts/sec)")
print(f"Embeddings shape: {all_embeddings.shape}")

## 4. Write Embeddings to Cloud SQL

In [ ]:
from tqdm.auto import tqdm

conn = get_conn()
cur = conn.cursor()

update_sql = "UPDATE products SET embedding = %s::vector WHERE parent_asin = %s"
write_batch = 500

for i in tqdm(range(0, len(asins), write_batch), desc="Writing to DB"):
    batch_asins = asins[i:i + write_batch]
    batch_embs = all_embeddings[i:i + write_batch]

    data = [(str(emb.tolist()), asin) for asin, emb in zip(batch_asins, batch_embs)]
    cur.executemany(update_sql, data)
    conn.commit()

cur.close()
conn.close()
print(f"Done! Wrote {len(asins)} embeddings.")

## 5. Verify & Create Index

In [ ]:
# Verify
conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM products WHERE embedding IS NOT NULL")
count = cur.fetchone()[0]
print(f"Products with embeddings: {count}")
cur.close()
conn.close()

In [ ]:
# Create IVFFlat index for fast cosine similarity search
# This takes ~1-2 minutes on 112K products
conn = get_conn()
cur = conn.cursor()
print("Creating IVFFlat index (this takes a minute)...")
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_products_embedding
    ON products USING ivfflat (embedding vector_cosine_ops)
    WITH (lists = 100)
""")
conn.commit()
cur.close()
conn.close()
print("Index created! Your database is ready for vector search.")

In [ ]:
# Clean up
connector.close()
print("All done! Your Cloud Run service should now return recommendations.")